In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import mlflow
import mlflow.sklearn

# Load dataset
df = pd.read_csv("../data/processed/heart_disease_clean.csv")

# Convert target column
df["target"] = df["num"].apply(lambda x: 1 if x > 0 else 0)
df.drop(columns=["num"], inplace=True)

# Encode categorical columns
label_enc = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = label_enc.fit_transform(df[col])

# Split data
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
}

mlflow.set_tracking_uri("sqlite:///d:/Bits_Pilani_Mtech/Semester3/Courses/MLOps/2025ae05403_Assignment1/mlops_heart_disease/notebooks/mlflow.db")
mlflow.set_experiment("HeartDiseasePrediction")

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_pred)

        print(f"\n{name} Results:")
        print("Accuracy:", acc)
        print("Precision:", prec)
        print("Recall:", rec)
        print("ROC‑AUC:", roc)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("roc_auc", roc)
        mlflow.sklearn.log_model(
            model,
            "model",
            skops_trusted_types=["sklearn.tree._tree.Tree"]  # explicitly trust tree type
        )


C:\Users\suren\AppData\Local\Temp\ipykernel_18452\1602255595.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
2026/10/20 19:51:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



LogisticRegression Results:
Accuracy: 0.8666666666666667
Precision: 0.8333333333333334
Recall: 0.8333333333333334
ROC‑AUC: 0.8611111111111112


2026/10/20 19:51:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



RandomForest Results:
Accuracy: 0.8833333333333333
Precision: 0.84
Recall: 0.875
ROC‑AUC: 0.8819444444444444
